# 교통표지판 43클래스 — EfficientNet-V2-S + 트랙 인지 교차검증

**이 노트북이 이전 버전과 다른 단 하나의 핵심**: 학습/검증 분할을 이미지 단위가 아니라
**트랙(촬영 시퀀스) 단위**로 합니다.

파일명 `클래스_트랙_프레임.png`에서 30프레임은 **같은 물리적 표지판 하나**를 찍은 것입니다.
이미지 단위로 랜덤 분할하면 같은 표지판의 다른 프레임이 train과 val에 동시에 들어가
검증 점수가 99.99%로 부풀려집니다. 그 상태에서는 "무엇이 개선인지"를 판정할 수 없습니다.

| | 이미지 단위 분할 (이전) | 트랙 단위 분할 (이 노트북) |
|---|---|---|
| val 점수 | 99.99% (거짓) | 97~98%대 예상 (정직) |
| 의미 | 본 적 있는 표지판의 다른 프레임 | **처음 보는 표지판** |
| 튜닝 근거 | 없음 | OOF 26,010장 |

### 데이터의 진짜 크기

총 868트랙. 클래스별 트랙 수는 42 / 21 / 6 세 종류이고,
**43개 중 22개 클래스는 물리적 표지판이 단 6개뿐**입니다.
채점 지표가 Macro-F1이므로 이 22개 클래스가 점수의 절반을 결정합니다.
즉 승부처는 "6개만 보고 처음 보는 7번째를 맞히는 능력"입니다.

### 실행 순서

1. `MODE = 'ablation'` 으로 fold 0만 돌려 레시피 후보를 비교
2. 채택한 설정으로 `MODE = 'cv'` 로 5폴드 전체 학습 → OOF 진단 → `result.csv` 생성

A100 기준 대략적인 소요 시간 (EfficientNet-V2-S @ 224px, 25 epoch)

| | ablation (fold 0만) | cv (5폴드 전체) |
|---|---|---|
| 학습 | 약 25분 | 약 2시간 |
| 추론 + 진단 | — | 약 10분 |

In [ ]:
# ============================================================
#  0. 라이브러리
# ============================================================
from pathlib import Path
import os, sys, io, json, math, time, random, shutil, zipfile, copy, warnings, unicodedata
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import cv2
from PIL import Image, ImageFilter, ImageDraw

from sklearn.metrics import f1_score, accuracy_score, confusion_matrix

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.transforms import InterpolationMode
import torchvision.transforms.functional as TF
from torchvision.models import efficientnet_b0, efficientnet_b2, efficientnet_v2_s
from torchvision.models import (
    EfficientNet_B0_Weights, EfficientNet_B2_Weights, EfficientNet_V2_S_Weights,
)

warnings.filterwarnings('ignore', category=UserWarning)
IN_COLAB = 'google.colab' in sys.modules

# 중요: OpenCV가 자체 스레드를 쓰면 DataLoader 워커와 충돌해 학습이 멈추는(deadlock)
# 고전적인 문제가 있습니다. 증강에서 cv2를 쓰므로 반드시 꺼야 합니다.
cv2.setNumThreads(0)

print('PyTorch      :', torch.__version__)
print('CUDA 사용가능:', torch.cuda.is_available())
print('Colab 환경   :', IN_COLAB)

## 1. 실행 제어판

**여기만 바꾸면 됩니다.** 아래 셀 외에는 건드릴 필요가 없습니다.

`MODE`
- `'ablation'` — fold 0 하나만 학습합니다. 레시피 후보를 비교할 때 씁니다.
  결과가 `ablation_log.csv` 에 계속 누적되므로 여러 번 돌려서 표로 비교하세요.
- `'cv'` — 5폴드 전체를 학습하고 OOF 진단과 `result.csv` 까지 만듭니다.

**ablation 판정 규칙**: fold 0 하나의 차이가 **0.3%p 미만이면 "판정 불가"** 로 보고 기본값을
유지하세요. 희소 클래스의 val 트랙이 1~2개뿐이라 폴드 하나의 점수는 크게 흔들립니다.
확실히 이긴 후보 1~2개만 `'cv'` 로 확정하는 것이 안전합니다.

In [ ]:
# ============================================================
#  1. 실행 제어판
# ============================================================
MODE    = 'cv'              # 'ablation' (fold 0만) | 'cv' (5폴드 + 제출)
RUN_TAG = 'v2s_224_base'    # ablation 로그에 남길 이름

# --- 모델 / 입력 ------------------------------------------------
BACKBONE   = 'efficientnet_v2_s'  # 'efficientnet_v2_s' | 'efficientnet_b2' | 'efficientnet_b0'
IMAGE_SIZE = 224                  # ablation 후보: 224 / 300 (V2-S는 384 평가 기준으로 학습됨)
USE_CLAHE  = False                # ablation 후보: False / True (대비 정규화)

# --- 학습 -------------------------------------------------------
EPOCHS        = 25
WARMUP_EPOCHS = 3
BATCH_SIZE    = 128     # A100 40GB 기준. OOM이면 96 -> 64 순으로 낮추세요.
                        # 300px로 올릴 때는 64 정도가 안전합니다.
LR            = 3e-4
WEIGHT_DECAY  = 1e-4     # ablation 후보: 1e-4 / 1e-2
LABEL_SMOOTH  = 0.10
EMA_DECAY     = 0.999
GRAD_CLIP     = 5.0
DROPOUT       = 0.30

USE_CUTMIX   = False     # ablation 후보. V2-S는 stochastic depth가 내장돼 정규화가 이미
                         # 한 겹 더 걸려 있어, 이 데이터 규모에선 과할 가능성이 큽니다.
CUTMIX_P     = 0.30
CUTMIX_ALPHA = 1.0

# --- 분할 -------------------------------------------------------
NUM_CLASSES = 43
N_SPLITS    = 5
SEED        = 42

# --- 안전장치 ---------------------------------------------------
# 폴드가 끝날 때마다 체크포인트를 Drive에 복사하고, 재실행하면 이미 끝난 폴드는
# 다시 학습하지 않고 불러옵니다. Colab 세션이 끊겨도 진행분을 잃지 않습니다.
# 설정(백본/해상도/CLAHE)이 저장된 것과 다르면 자동으로 다시 학습합니다.
RESUME = True

# --- 추론 -------------------------------------------------------
USE_TTA     = True
TUNE_TAU    = True       # Macro-F1용 logit adjustment 계수를 OOF에서 탐색
INFER_BATCH = 256

# --- 경로 -------------------------------------------------------
# Google Drive의 프로젝트 폴더. 이 안에서 Train/Test 폴더나 ZIP을 찾습니다.
# Colab 좌측 파일 탐색기에서 폴더 우클릭 > "경로 복사" 하면 정확한 값을 얻습니다.
DRIVE_PROJECT_ROOT = (
    '/content/drive/MyDrive/데이터 사이언티스트(서울대)/02. 강의자료/'
    'Deep Learning_강유교수님/프로젝트실습1/Team Project'
)

# 자동 탐색이 실패할 때만 데이터 폴더(Train/Test의 부모)를 직접 지정하세요.
MANUAL_DATA_ROOT = None  # 예: '/content/dataset_local/data/data 2'

assert MODE in ('ablation', 'cv'), "MODE는 'ablation' 또는 'cv'"
print(f'MODE={MODE} | {BACKBONE} @ {IMAGE_SIZE}px | CLAHE={USE_CLAHE} | '
      f'CutMix={USE_CUTMIX} | wd={WEIGHT_DECAY} | epochs={EPOCHS}')

In [ ]:
# ============================================================
#  2. 재현성과 디바이스
# ============================================================
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True      # 입력 크기가 고정이라 켜두면 빠릅니다

set_seed()

if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU를 찾지 못했습니다.\n'
        'Colab: 런타임 > 런타임 유형 변경 > 하드웨어 가속기를 GPU로 설정하세요.'
    )

DEVICE = torch.device('cuda')
_props = torch.cuda.get_device_properties(0)
print(f'Device: {_props.name} | VRAM {_props.total_memory / 2**30:.1f} GB | CUDA {torch.version.cuda}')

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# torch 2.4 부터 AMP API가 바뀌어 경고가 나옵니다. 버전에 맞는 쪽을 씁니다.
_TV = tuple(int(x) for x in torch.__version__.split('.')[:2])
_NEW_AMP = _TV >= (2, 4)

def amp_autocast():
    return torch.amp.autocast('cuda') if _NEW_AMP else torch.cuda.amp.autocast(enabled=True)

def make_scaler():
    return torch.amp.GradScaler('cuda') if _NEW_AMP else torch.cuda.amp.GradScaler(enabled=True)

## 2. 경로 탐색

Drive의 프로젝트 폴더 안에서 `Train/`(0~42)과 `Test/`가 함께 있는 곳을 찾습니다.

**두 가지 함정을 피해야 합니다.**

**(1) 한글 폴더명의 NFC / NFD**
macOS에서 만든 폴더는 한글이 **NFD(자모 분리)** 로 저장됩니다.
`'강의자료'` 가 `ㄱ ㅏ ㅇ ...` 처럼요. 코드에 적는 문자열은 **NFC(완성형)** 입니다.
화면에는 똑같이 보이지만 `==` 비교는 실패해서 "폴더가 없다"는 엉뚱한 에러가 납니다.
경로를 한 단계씩 내려가며 정규화해서 맞춥니다.

**(2) 파일시스템 전체 탐색 금지**
`/` 같은 상위에서 `rglob` 을 돌리면 `/proc` 같은 커널 가상 폴더에 닿아
`OSError: [Errno 22]` 로 죽습니다. 깊이 제한을 **탐색기 안쪽**에 넣은 너비우선 탐색을 쓰고,
시스템 폴더는 아예 건너뜁니다.

**(3) Drive에서 직접 학습하지 않기**
Drive는 작은 파일 26,010개를 읽을 때 극도로 느립니다. `/content` 로컬 디스크로
복사한 뒤 학습합니다. 복사에 1~3분 걸리지만 매 epoch마다 그 이상을 아낍니다.

In [ ]:
# ============================================================
#  3. 경로 탐색
# ============================================================
IMAGE_EXTS = {'.png', '.jpg', '.jpeg', '.ppm', '.bmp'}

# 시스템/부산물 폴더는 탐색에서 제외 (/proc 등을 훑으면 OSError로 죽습니다)
SKIP_DIR_NAMES = {
    'proc', 'sys', 'dev', 'run', 'boot', 'bin', 'sbin', 'lib', 'lib32', 'lib64',
    'usr', 'etc', 'var', 'opt', 'snap', 'srv', 'mnt', 'media',
    '__pycache__', 'node_modules', '.ipynb_checkpoints', '__MACOSX',
}

def looks_like_dataset(root):
    # Train/0..42 가 모두 채워져 있고 Test/ 에 이미지가 있으면 진짜 데이터 루트
    try:
        root = Path(root)
        train, test = root / 'Train', root / 'Test'
        if not (train.is_dir() and test.is_dir()):
            return False
        for class_id in range(NUM_CLASSES):
            class_dir = train / str(class_id)
            if not class_dir.is_dir():
                return False
            if not any(p.suffix.lower() in IMAGE_EXTS for p in class_dir.iterdir()):
                return False
        return any(p.suffix.lower() in IMAGE_EXTS for p in test.iterdir())
    except OSError:
        return False          # 권한 없음 / 끊어진 링크 / 가상 파일시스템

def search_dataset(start, max_depth=5):
    # 너비우선 탐색. 깊이 제한을 탐색기 안쪽에서 적용해 실제로 탐색을 멈춥니다.
    start = Path(start)
    if looks_like_dataset(start):
        return start.resolve()
    frontier, visited = [(start, 0)], set()
    while frontier:
        current, depth = frontier.pop(0)
        if depth >= max_depth:
            continue
        try:
            children = sorted(p for p in current.iterdir() if p.is_dir())
        except OSError:
            continue
        for child in children:
            if child.name.startswith('.') or child.name in SKIP_DIR_NAMES:
                continue
            key = str(child)
            if key in visited:
                continue
            visited.add(key)
            if looks_like_dataset(child):
                return child.resolve()
            frontier.append((child, depth + 1))
    return None

def resolve_korean_path(path_str):
    # 경로를 한 단계씩 내려가며 유니코드 정규화(NFC)로 비교해 실제 폴더를 찾습니다.
    parts = Path(path_str).parts
    current = Path(parts[0])
    for wanted in parts[1:]:
        target = unicodedata.normalize('NFC', wanted)
        try:
            match = next((c for c in current.iterdir()
                          if unicodedata.normalize('NFC', c.name) == target), None)
        except OSError:
            return None
        if match is None:
            return None
        current = match
    return current.resolve()

# --- 프로젝트 루트 ----------------------------------------------
if IN_COLAB:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').is_dir():
        drive.mount('/content/drive')

    PROJECT_ROOT = resolve_korean_path(DRIVE_PROJECT_ROOT)
    if PROJECT_ROOT is None:
        raise FileNotFoundError(
            f'Drive에서 프로젝트 폴더를 찾지 못했습니다:\n  {DRIVE_PROJECT_ROOT}\n\n'
            '제어판 셀의 DRIVE_PROJECT_ROOT 를 실제 경로로 고쳐주세요.\n'
            'Colab 좌측 파일 탐색기에서 폴더를 우클릭 > "경로 복사" 하면 정확합니다.'
        )
else:
    PROJECT_ROOT = Path.cwd().resolve()

print('Project root:', PROJECT_ROOT)

# --- 데이터 루트 ------------------------------------------------
LOCAL_DATA_DIR = Path('/content/dataset_local')
DATA_ROOT = None

if MANUAL_DATA_ROOT:                                   # 직접 지정이 최우선
    DATA_ROOT = Path(MANUAL_DATA_ROOT)
    if not looks_like_dataset(DATA_ROOT):
        raise FileNotFoundError(f'MANUAL_DATA_ROOT에 Train/Test가 없습니다: {DATA_ROOT}')
    DATA_ROOT = DATA_ROOT.resolve()

if DATA_ROOT is None:                                  # 이전 실행에서 풀어둔 로컬 사본
    DATA_ROOT = search_dataset(LOCAL_DATA_DIR, max_depth=4)
    if DATA_ROOT is not None:
        print('기존 로컬 사본을 재사용합니다.')

if DATA_ROOT is None:                                  # 프로젝트 폴더 안을 탐색
    print('프로젝트 폴더에서 Train/Test를 찾는 중...')
    DATA_ROOT = search_dataset(PROJECT_ROOT, max_depth=5)

if DATA_ROOT is None:                                  # 없으면 ZIP을 찾아 해제
    archives = sorted(
        (p for p in PROJECT_ROOT.rglob('*.zip')
         if len(p.relative_to(PROJECT_ROOT).parts) <= 3),
        key=lambda p: -p.stat().st_size,
    )
    if not archives:
        raise FileNotFoundError(
            f'프로젝트 폴더에서 Train/Test 폴더도, ZIP 파일도 찾지 못했습니다.\n'
            f'  탐색 위치: {PROJECT_ROOT}\n'
            '제어판 셀의 MANUAL_DATA_ROOT 에 데이터 폴더를 직접 적어주세요.'
        )
    print(f'ZIP을 해제합니다: {archives[0].name} ({archives[0].stat().st_size / 2**20:.0f}MB)')
    if LOCAL_DATA_DIR.exists():
        shutil.rmtree(LOCAL_DATA_DIR)
    LOCAL_DATA_DIR.mkdir(parents=True)
    with zipfile.ZipFile(archives[0]) as zf:
        base = LOCAL_DATA_DIR.resolve()
        for member in zf.infolist():                   # zip slip 방지
            target = (base / member.filename.replace('\\', '/')).resolve()
            if base != target and base not in target.parents:
                raise RuntimeError(f'안전하지 않은 압축 경로: {member.filename}')
        zf.extractall(base)
    DATA_ROOT = search_dataset(LOCAL_DATA_DIR, max_depth=4)
    if DATA_ROOT is None:
        raise RuntimeError('ZIP을 풀었지만 0~42 클래스가 모두 채워진 Train/Test를 찾지 못했습니다.')

# --- Drive에 있으면 로컬 디스크로 복사 (학습 속도의 핵심) --------
if IN_COLAB and str(DATA_ROOT).startswith('/content/drive'):
    if not looks_like_dataset(LOCAL_DATA_DIR):
        print('Drive는 작은 파일 읽기가 매우 느립니다. /content 로 복사합니다...')
        t0 = time.time()
        if LOCAL_DATA_DIR.exists():
            shutil.rmtree(LOCAL_DATA_DIR)
        shutil.copytree(DATA_ROOT, LOCAL_DATA_DIR)
        print(f'복사 완료 ({time.time() - t0:.0f}초)')
    DATA_ROOT = LOCAL_DATA_DIR.resolve()

TRAIN_DIR = DATA_ROOT / 'Train'
TEST_DIR  = DATA_ROOT / 'Test'

# --- 출력 경로 --------------------------------------------------
# 결과물은 Drive에 (런타임이 끊겨도 남도록), 체크포인트는 /content에 (빠르게)
OUTPUT_DIR = PROJECT_ROOT / 'trackcv_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR = Path('/content/checkpoints') if IN_COLAB else (OUTPUT_DIR / 'checkpoints')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

# 제출 템플릿(파일명 순서의 기준). 없으면 Test 폴더를 정렬해 씁니다.
RESULT_TEMPLATE = next(
    (p for p in [PROJECT_ROOT / 'result_template_before_prediction.csv',
                 PROJECT_ROOT / 'result.csv'] if p.is_file()),
    None,
)

print('Data root :', DATA_ROOT)
print('  Train   :', sum(1 for _ in TRAIN_DIR.rglob('*.png')), '장')
print('  Test    :', sum(1 for _ in TEST_DIR.glob('*.png')), '장')
print('Output    :', OUTPUT_DIR)
print('Checkpoint:', CKPT_DIR)
print('Template  :', RESULT_TEMPLATE if RESULT_TEMPLATE else '(없음 - Test 정렬 순서 사용)')

## 3. 트랙 파싱 — 이 노트북의 핵심

파일명 `00012_00003_00017.png` = **클래스 12 / 트랙 3 / 프레임 17**.

트랙 번호는 클래스마다 0부터 다시 시작하므로, 그룹 키는 반드시
`"클래스_트랙"` 형태여야 합니다. 트랙 번호만 쓰면 서로 다른 클래스의 트랙이
같은 그룹으로 묶여버립니다.

In [ ]:
# ============================================================
#  4. 트랙 파싱
# ============================================================
def build_sample_table(train_dir):
    rows = []
    for class_id in range(NUM_CLASSES):
        class_dir = Path(train_dir) / str(class_id)
        for path in sorted(class_dir.iterdir()):
            if path.suffix.lower() not in IMAGE_EXTS:
                continue                      # .DS_Store 같은 부산물 무시
            parts = path.stem.split('_')
            if len(parts) != 3:
                raise ValueError(f'예상과 다른 파일명입니다: {path.name}')
            file_class, track, frame = (int(x) for x in parts)
            if file_class != class_id:
                raise ValueError(f'폴더({class_id})와 파일명({file_class})의 클래스가 다릅니다: {path.name}')
            rows.append({
                'path':  str(path),
                'label': class_id,
                'track': track,
                'frame': frame,
                'group': f'{class_id}_{track}',   # 클래스마다 트랙 번호가 재사용되므로 결합
            })
    return pd.DataFrame(rows)

samples = build_sample_table(TRAIN_DIR)
paths   = samples['path'].to_numpy()
labels  = samples['label'].to_numpy()
groups  = samples['group'].to_numpy()
N       = len(samples)

tracks_per_class = samples.groupby('label')['group'].nunique()
imgs_per_class   = samples.groupby('label').size()

print(f'학습 이미지 {N:,}장 / 트랙 {samples["group"].nunique():,}개 / 클래스 {samples["label"].nunique()}개')
print()
print('클래스별 트랙 수 분포:')
for n_tracks, n_classes in sorted(Counter(tracks_per_class).items()):
    members = sorted(tracks_per_class[tracks_per_class == n_tracks].index.tolist())
    print(f'  트랙 {n_tracks:2d}개 → 클래스 {n_classes:2d}개  {members if n_classes <= 25 else ""}')
print()
print(f'>> 트랙 6개뿐인 희소 클래스 {int((tracks_per_class == 6).sum())}개가 Macro-F1의 절반을 좌우합니다.')

assert samples['label'].nunique() == NUM_CLASSES, '43개 클래스가 모두 있어야 합니다'
assert N > 0

In [ ]:
# ============================================================
#  5. 한글 표지판 이름 (진단 리포트용)
# ============================================================
SIGN_NAMES_KO = {
    0: '제한속도 20', 1: '제한속도 30', 2: '제한속도 50', 3: '제한속도 60',
    4: '제한속도 70', 5: '제한속도 80', 6: '제한속도 80 해제', 7: '제한속도 100',
    8: '제한속도 120', 9: '추월 금지', 10: '3.5톤 초과 추월 금지',
    11: '다음 교차로 우선권', 12: '우선도로', 13: '양보', 14: '정지',
    15: '차량 통행 금지', 16: '3.5톤 초과 통행 금지', 17: '진입 금지',
    18: '일반 위험 주의', 19: '좌측 위험 커브', 20: '우측 위험 커브',
    21: '연속 위험 커브', 22: '요철 도로', 23: '미끄러운 도로',
    24: '우측 차로 좁아짐', 25: '도로 공사', 26: '신호등 주의',
    27: '보행자 주의', 28: '어린이 보호', 29: '자전거 주의',
    30: '빙판·눈길 주의', 31: '야생동물 주의', 32: '속도·추월 제한 해제',
    33: '우회전 지시', 34: '좌회전 지시', 35: '직진 지시',
    36: '직진 또는 우회전', 37: '직진 또는 좌회전', 38: '우측 통행',
    39: '좌측 통행', 40: '회전교차로', 41: '추월 금지 해제',
    42: '3.5톤 초과 추월 금지 해제',
}
assert len(SIGN_NAMES_KO) == NUM_CLASSES

## 4. 트랙 단위 5폴드 분할 + 누수 검사

### 왜 `StratifiedGroupKFold` 를 쓰지 않는가

sklearn의 `StratifiedGroupKFold` 는 **각 폴드에 모든 클래스가 들어간다는 보장을 하지 않습니다.**
전체 클래스 비율의 불균형을 탐욕적으로 줄일 뿐이라, 트랙이 6개뿐인 클래스가
`2, 2, 2, 0, 0` 으로 떨어져도 그대로 통과시킵니다.
val에 없는 클래스가 생기면 Macro-F1이 정의되지 않습니다.

### 대신 쓰는 방법 — 클래스별 라운드로빈

> 클래스마다 트랙을 섞은 뒤, 폴드에 하나씩 돌아가며 나눠줍니다.

트랙 수가 폴드 수 이상인 클래스는 **폴드마다 최소 1개를 자동으로 받습니다.**

| 클래스 유형 | 트랙 수 | 폴드별 배분 |
|---|---|---|
| 희소 22개 | 6 | 2, 1, 1, 1, 1 |
| 중간 7개 | 21 | 5, 4, 4, 4, 4 |
| 다수 14개 | 42 | 9, 9, 8, 8, 8 |

라운드로빈 시작 위치를 클래스 사이에 이어가므로 나머지 트랙이 fold 0에만 몰리지 않고
폴드 크기도 고르게 맞습니다.

### 강제하는 세 가지

하나라도 깨지면 이후 모든 숫자가 무의미해지므로 `assert` 로 막습니다.

1. 같은 트랙이 train과 val에 동시에 등장하지 않을 것 (**누수 금지**)
2. 모든 폴드의 val에 43개 클래스가 전부 존재할 것
3. 모든 이미지가 정확히 한 번씩 val이 될 것 (OOF 구성의 전제)

In [ ]:
# ============================================================
#  6. 트랙 단위 분할
# ============================================================
def make_track_folds(labels, groups, n_splits, seed):
    # 클래스별로 트랙을 섞어 폴드에 라운드로빈 배정합니다.
    # offset을 클래스 사이에 이어받아, 나머지 트랙이 특정 폴드에 몰리지 않게 합니다.
    rng = np.random.RandomState(seed)
    track_label = dict(zip(groups, labels))
    fold_of_track, offset = {}, 0
    for class_id in sorted(set(labels.tolist())):
        tracks = sorted(track for track, y in track_label.items() if y == class_id)
        rng.shuffle(tracks)
        for i, track in enumerate(tracks):
            fold_of_track[track] = (offset + i) % n_splits
        offset = (offset + len(tracks)) % n_splits
    assignment = np.array([fold_of_track[g] for g in groups])
    return [(np.where(assignment != k)[0], np.where(assignment == k)[0])
            for k in range(n_splits)]

# 폴드 수보다 트랙이 적은 클래스가 있으면 폴드마다 1개씩 줄 수 없습니다.
min_tracks = int(tracks_per_class.min())
assert min_tracks >= N_SPLITS, (
    f'가장 트랙이 적은 클래스의 트랙이 {min_tracks}개뿐입니다. '
    f'N_SPLITS를 {min_tracks} 이하로 낮추세요.'
)

folds = make_track_folds(labels, groups, N_SPLITS, SEED)

print(f'{N_SPLITS}폴드 분할 검사')
print('-' * 72)
for k, (tr_idx, va_idx) in enumerate(folds):
    tr_groups, va_groups = set(groups[tr_idx]), set(groups[va_idx])

    # (1) 트랙 누수 검사 — 이 노트북 전체의 전제
    overlap = tr_groups & va_groups
    assert not overlap, f'fold {k}: 트랙 누수 발생! {sorted(overlap)[:5]}'

    # (2) val에 43개 클래스가 모두 있는지 — 라운드로빈 배정이라 원래 깨질 수 없습니다.
    missing = sorted(set(range(NUM_CLASSES)) - set(labels[va_idx]))
    assert not missing, f'fold {k}: val에 없는 클래스 {missing} (분할 로직 버그)'

    rare_val = sum(1 for g in va_groups if tracks_per_class[int(g.split('_')[0])] == 6)
    print(f'  fold {k}: train {len(tr_idx):6,}장 / {len(tr_groups):3d}트랙  |  '
          f'val {len(va_idx):5,}장 / {len(va_groups):3d}트랙  (희소클래스 트랙 {rare_val}개)')

# 모든 이미지가 정확히 한 번씩 val이 되는지 (OOF의 전제)
val_counts = np.zeros(N, dtype=int)
for _, va_idx in folds:
    val_counts[va_idx] += 1
assert (val_counts == 1).all(), 'OOF 불가: 모든 이미지가 정확히 한 번 val이어야 합니다'

print('-' * 72)
print('PASS: 트랙 누수 없음 / 전 클래스 존재 / OOF 커버리지 완전')

## 5. 전처리와 증강

### 왜 이런 증강인가

트랙 안의 30프레임은 이미 **거리·블러·흔들림 변화**를 담고 있습니다.
그쪽 증강을 아무리 세게 걸어도 데이터에 이미 있는 정보가 중복될 뿐입니다.

반대로 **"같은 종류지만 다른 위치에 설치된, 다르게 낡은 표지판"** 은 데이터에 아예 없습니다.
희소 클래스는 표지판 개체가 6개뿐이니까요. 이 축은 증강으로만 만들 수 있습니다.
그래서 `RandomPerspective`(설치 각도)와 `ColorJitter`(색바램·조명)의 비중을 높였습니다.

### 좌우 반전 금지

```
33 우회전 ↔ 34 좌회전      19 좌측커브 ↔ 20 우측커브
38 우측통행 ↔ 39 좌측통행   36 직진+우 ↔ 37 직진+좌
```
좌우로 뒤집으면 **라벨이 바뀝니다.** 절대 넣으면 안 됩니다.

In [ ]:
# ============================================================
#  7. 전처리 / 증강
# ============================================================
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
RESAMPLE = getattr(Image, 'Resampling', Image)

_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))

class CLAHE:
    # 저노출 이미지가 많은 GTSRB의 고전적 전처리.
    # LAB의 밝기(L) 채널만 국소 히스토그램 평활화 → 색상은 보존됩니다.
    def __call__(self, image):
        lab = cv2.cvtColor(np.asarray(image), cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        merged = cv2.merge((_clahe.apply(l), a, b))
        return Image.fromarray(cv2.cvtColor(merged, cv2.COLOR_LAB2RGB))

class RandomLowResolution:
    # 멀리서 찍혀 뭉개진 표지판을 모사
    def __init__(self, p=0.35, scale=(0.25, 0.70)):
        self.p, self.scale = p, scale
    def __call__(self, image):
        if random.random() >= self.p:
            return image
        w, h = image.size
        r = random.uniform(*self.scale)
        small = (max(8, int(w * r)), max(8, int(h * r)))
        down = random.choice([RESAMPLE.BILINEAR, RESAMPLE.BOX])
        up   = random.choice([RESAMPLE.NEAREST, RESAMPLE.BILINEAR, RESAMPLE.BICUBIC])
        return image.resize(small, down).resize((w, h), up)

class RandomJPEG:
    # 압축 아티팩트 모사
    def __init__(self, p=0.25, quality=(25, 80)):
        self.p, self.quality = p, quality
    def __call__(self, image):
        if random.random() >= self.p:
            return image
        buf = io.BytesIO()
        image.save(buf, format='JPEG', quality=random.randint(*self.quality))
        buf.seek(0)
        with Image.open(buf) as decoded:
            return decoded.convert('RGB')

class RandomMotionBlur:
    # 주행 중 흔들림 모사
    def __init__(self, p=0.18, kernels=(3, 5, 7), angle=(-45, 45)):
        self.p, self.kernels, self.angle = p, kernels, angle
    def __call__(self, image):
        if random.random() >= self.p:
            return image
        k = random.choice(self.kernels)
        kernel = np.zeros((k, k), dtype=np.float32)
        kernel[k // 2, :] = 1.0
        m = cv2.getRotationMatrix2D((k / 2 - 0.5, k / 2 - 0.5), random.uniform(*self.angle), 1.0)
        kernel = cv2.warpAffine(kernel, m, (k, k))
        kernel /= max(kernel.sum(), 1e-8)
        return Image.fromarray(cv2.filter2D(np.asarray(image), -1, kernel))

class TTAAffine:
    # 추론 시 결정적(랜덤 아님) 확대/회전
    def __init__(self, scale=1.0, angle=0.0):
        self.scale, self.angle = scale, angle
    def __call__(self, image):
        return TF.affine(image, angle=self.angle, translate=(0, 0),
                         scale=self.scale, shear=[0.0, 0.0],
                         interpolation=InterpolationMode.BILINEAR)

def build_train_transform(size, use_clahe):
    ops = [transforms.Resize((size, size), interpolation=InterpolationMode.BICUBIC, antialias=True)]
    if use_clahe:
        ops.append(CLAHE())
    ops += [
        # 개체 다양성 축 — 여기가 희소 클래스 일반화의 핵심
        transforms.RandomApply([transforms.RandomAffine(
            degrees=15, translate=(0.10, 0.10), scale=(0.80, 1.20), shear=8,
            interpolation=InterpolationMode.BILINEAR)], p=0.80),
        transforms.RandomPerspective(distortion_scale=0.20, p=0.35),
        transforms.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.25, hue=0.03),
        # 촬영 조건 축 — 트랙 내 프레임이 이미 일부 담고 있어 약하게만
        RandomLowResolution(p=0.30, scale=(0.28, 0.75)),
        RandomJPEG(p=0.22, quality=(30, 85)),
        RandomMotionBlur(p=0.15),
        transforms.RandomApply([transforms.GaussianBlur(3, sigma=(0.1, 1.5))], p=0.12),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        transforms.RandomErasing(p=0.25, scale=(0.02, 0.12), ratio=(0.4, 2.5), value='random'),
    ]
    # 주의: 좌우 반전(RandomHorizontalFlip)은 라벨을 바꾸므로 절대 넣지 않습니다.
    return transforms.Compose(ops)

def build_eval_transform(size, use_clahe, scale=1.0, angle=0.0):
    ops = [transforms.Resize((size, size), interpolation=InterpolationMode.BICUBIC, antialias=True)]
    if use_clahe:
        ops.append(CLAHE())
    if scale != 1.0 or angle != 0.0:
        ops.append(TTAAffine(scale, angle))
    ops += [transforms.ToTensor(), transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)]
    return transforms.Compose(ops)

# TTA 조합: 확대/축소 3 + 소각도 회전 2. 좌우 반전은 없습니다.
TTA_SPECS = [(1.00, 0.0), (0.92, 0.0), (1.08, 0.0), (1.00, -7.0), (1.00, 7.0)]

train_tf = build_train_transform(IMAGE_SIZE, USE_CLAHE)
eval_tf  = build_eval_transform(IMAGE_SIZE, USE_CLAHE)

# 변환이 실제로 3채널 유한 텐서를 만드는지 즉시 확인
_probe = Image.new('RGB', (48, 52), (120, 100, 80))
for _name, _tf in [('train', train_tf), ('eval', eval_tf)]:
    _out = _tf(_probe)
    assert _out.shape == (3, IMAGE_SIZE, IMAGE_SIZE) and torch.isfinite(_out).all(), _name
print(f'PASS: 증강 자체검사 (출력 {tuple(_out.shape)}, TTA {len(TTA_SPECS)}조합)')

In [ ]:
# ============================================================
#  8. Dataset / DataLoader
# ============================================================
NUM_WORKERS = min(8, os.cpu_count() or 2) if os.name != 'nt' else 0

class TrafficSignDataset(Dataset):
    # indices로 원본 배열의 부분집합을 가리킵니다. transform만 바꿔 재사용합니다.
    def __init__(self, file_paths, targets, indices, transform):
        self.file_paths = file_paths
        self.targets = targets
        self.indices = np.asarray(indices)
        self.transform = transform
    def __len__(self):
        return len(self.indices)
    def __getitem__(self, i):
        j = int(self.indices[i])
        with Image.open(self.file_paths[j]) as image:
            tensor = self.transform(image.convert('RGB'))
        target = -1 if self.targets is None else int(self.targets[j])
        return tensor, target

def make_loader(dataset, batch_size, shuffle, drop_last=False):
    kwargs = dict(batch_size=batch_size, shuffle=shuffle, drop_last=drop_last,
                  num_workers=NUM_WORKERS, pin_memory=True)
    if NUM_WORKERS > 0:
        kwargs.update(persistent_workers=True, prefetch_factor=4)
    return DataLoader(dataset, **kwargs)

print(f'DataLoader workers: {NUM_WORKERS}')

## 6. 모델 · 학습 루프

- **EMA(지수이동평균)**: 희소 클래스는 val 트랙이 1~2개뿐이라 epoch마다 점수가 크게 튑니다.
  EMA 없이 "best epoch"를 고르면 사실상 노이즈를 고르는 셈이라, 평활화된 가중치로 평가합니다.
- **warmup 3 + cosine**: pretrained 백본을 초반 큰 LR로 망가뜨리지 않기 위함입니다.
- **클래스 가중치**: 이미지 수의 역제곱근. 완전 역수는 6트랙 클래스에 7배 가중이 걸려
  학습이 불안정해집니다. 제곱근이 절충안입니다.

In [ ]:
# ============================================================
#  9. 모델 / EMA / 스케줄러
# ============================================================
# 세 백본 모두 classifier = Sequential(Dropout, Linear) 구조라 헤드 교체 코드가 동일합니다.
#   efficientnet_v2_s : 21.5M 파라미터, 마지막 Linear in_features=1280
#   efficientnet_b2   :  9.1M 파라미터, in_features=1408
#   efficientnet_b0   :  5.3M 파라미터, in_features=1280
BACKBONES = {
    'efficientnet_v2_s': (efficientnet_v2_s, EfficientNet_V2_S_Weights.IMAGENET1K_V1),
    'efficientnet_b2':   (efficientnet_b2,   EfficientNet_B2_Weights.IMAGENET1K_V1),
    'efficientnet_b0':   (efficientnet_b0,   EfficientNet_B0_Weights.IMAGENET1K_V1),
}

def build_model():
    factory, weights = BACKBONES[BACKBONE]
    model = factory(weights=weights)
    in_features = model.classifier[-1].in_features   # 마지막 Linear의 입력 차원
    model.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(in_features, NUM_CLASSES))
    return model.to(DEVICE)

@torch.no_grad()
def update_ema(ema_model, model, decay):
    ema_state, model_state = ema_model.state_dict(), model.state_dict()
    for key, ema_value in ema_state.items():
        source = model_state[key].detach()
        if ema_value.dtype.is_floating_point:
            ema_value.mul_(decay).add_(source, alpha=1.0 - decay)
        else:
            ema_value.copy_(source)          # BN의 num_batches_tracked 등 정수 버퍼

def make_scheduler(optimizer, total_steps, warmup_steps, min_ratio=0.02):
    def lr_lambda(step):
        if step < warmup_steps:
            return (step + 1) / max(1, warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        return min_ratio + (1 - min_ratio) * 0.5 * (1 + math.cos(math.pi * progress))
    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def compute_class_weights(train_indices):
    counts = np.bincount(labels[train_indices], minlength=NUM_CLASSES).astype(np.float64)
    assert (counts > 0).all(), f'학습 데이터가 없는 클래스: {np.where(counts == 0)[0].tolist()}'
    weights = np.sqrt(counts.sum() / (NUM_CLASSES * counts))
    weights = weights / weights.mean()
    return torch.tensor(weights, dtype=torch.float32, device=DEVICE)

def cutmix_batch(x, y, alpha):
    # 두 이미지를 사각형으로 섞고, 라벨은 면적 비율로 두 손실을 섞습니다.
    lam = float(np.random.beta(alpha, alpha))
    perm = torch.randperm(x.size(0), device=x.device)
    h, w = x.size(2), x.size(3)
    cut_h, cut_w = int(h * math.sqrt(1 - lam)), int(w * math.sqrt(1 - lam))
    cy, cx = np.random.randint(h), np.random.randint(w)
    y1, y2 = np.clip([cy - cut_h // 2, cy + cut_h // 2], 0, h)
    x1, x2 = np.clip([cx - cut_w // 2, cx + cut_w // 2], 0, w)
    x[:, :, y1:y2, x1:x2] = x[perm, :, y1:y2, x1:x2]
    lam = 1 - ((x2 - x1) * (y2 - y1) / (h * w))     # 실제 잘린 면적으로 보정
    return x, y, y[perm], lam

In [ ]:
# ============================================================
# 10. 학습 / 평가 함수
# ============================================================
@torch.no_grad()
def predict_probs(model, loader):
    # 소프트맥스 확률 (N, 43) 과 정답을 반환. 정답이 없으면 targets는 전부 -1.
    model.eval()
    probs, targets = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        with amp_autocast():
            logits = model(x)
        probs.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
        targets.append(y.numpy())
    return np.concatenate(probs), np.concatenate(targets)

def train_one_epoch(model, ema_model, loader, criterion, optimizer, scheduler, scaler):
    model.train()
    running_loss, seen = 0.0, 0
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        use_cutmix = USE_CUTMIX and random.random() < CUTMIX_P
        if use_cutmix:
            x, y_a, y_b, lam = cutmix_batch(x, y, CUTMIX_ALPHA)

        with amp_autocast():
            logits = model(x)
            if use_cutmix:
                loss = lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)
            else:
                loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        update_ema(ema_model, model, EMA_DECAY)

        running_loss += loss.item() * y.size(0)
        seen += y.size(0)
    return running_loss / max(seen, 1)

def fit_fold(fold, train_indices, val_indices):
    set_seed(SEED + fold)                  # 폴드마다 다른 시드 → 앙상블 다양성
    model = build_model()
    ema_model = copy.deepcopy(model).eval()
    for parameter in ema_model.parameters():
        parameter.requires_grad_(False)

    train_loader = make_loader(
        TrafficSignDataset(paths, labels, train_indices, train_tf),
        BATCH_SIZE, shuffle=True, drop_last=True)
    val_loader = make_loader(
        TrafficSignDataset(paths, labels, val_indices, eval_tf),
        INFER_BATCH, shuffle=False)

    criterion = nn.CrossEntropyLoss(weight=compute_class_weights(train_indices),
                                    label_smoothing=LABEL_SMOOTH)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    steps = len(train_loader)
    scheduler = make_scheduler(optimizer, EPOCHS * steps, WARMUP_EPOCHS * steps)
    scaler = make_scaler()

    checkpoint_path = CKPT_DIR / f'{BACKBONE}_{IMAGE_SIZE}_fold{fold}.pt'
    best_f1, best_state, history = -1.0, None, []

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        train_loss = train_one_epoch(model, ema_model, train_loader,
                                     criterion, optimizer, scheduler, scaler)
        val_probs, val_true = predict_probs(ema_model, val_loader)   # EMA 가중치로 평가
        val_pred = val_probs.argmax(axis=1)
        val_f1  = f1_score(val_true, val_pred, average='macro')
        val_acc = accuracy_score(val_true, val_pred)
        history.append({'epoch': epoch, 'train_loss': train_loss,
                        'val_macro_f1': val_f1, 'val_acc': val_acc,
                        'lr': optimizer.param_groups[0]['lr']})

        flag = ''
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = {k: v.detach().cpu().clone() for k, v in ema_model.state_dict().items()}
            # 값은 전부 파이썬 기본형으로 저장합니다. 그래야 나중에 불러올 때
            # weights_only=True (임의 코드 실행을 막는 안전 모드)로 읽을 수 있습니다.
            torch.save({'model': best_state, 'fold': int(fold), 'epoch': int(epoch),
                        'val_macro_f1': float(best_f1), 'backbone': str(BACKBONE),
                        'image_size': int(IMAGE_SIZE), 'use_clahe': bool(USE_CLAHE)},
                       checkpoint_path)
            flag = '  <-- best'

        print(f'  [fold {fold}] {epoch:02d}/{EPOCHS} | loss {train_loss:.4f} | '
              f'val acc {val_acc:.5f} macroF1 {val_f1:.5f} | {time.time() - t0:.0f}s{flag}')

    ema_model.load_state_dict(best_state)
    print(f'  [fold {fold}] best val Macro-F1 = {best_f1:.5f}  -> {checkpoint_path.name}')
    return ema_model, best_f1, pd.DataFrame(history), checkpoint_path

## 7. 학습 실행

`MODE='ablation'` 이면 fold 0만, `MODE='cv'` 면 5폴드 전부를 학습합니다.

**진행 상황을 보는 법**: val Macro-F1이 97~98% 부근에서 움직이면 정상입니다.
99.9%가 나온다면 트랙 누수가 되살아난 것이니 6번 셀의 assert를 다시 확인하세요.

In [ ]:
# ============================================================
# 11. 학습 실행
# ============================================================
active_folds = [0] if MODE == 'ablation' else list(range(N_SPLITS))
fold_models, fold_scores, histories = {}, {}, []

# 체크포인트 영구 보관소. /content는 세션이 끝나면 사라지므로 Drive에도 남깁니다.
DRIVE_CKPT_DIR = OUTPUT_DIR / 'checkpoints'
DRIVE_CKPT_DIR.mkdir(parents=True, exist_ok=True)

def checkpoint_matches_config(ckpt):
    return (ckpt.get('backbone') == BACKBONE
            and ckpt.get('image_size') == IMAGE_SIZE
            and ckpt.get('use_clahe') == USE_CLAHE)

overall_t0 = time.time()
for fold in active_folds:
    train_indices, val_indices = folds[fold]
    drive_ckpt = DRIVE_CKPT_DIR / f'{BACKBONE}_{IMAGE_SIZE}_fold{fold}.pt'

    # --- 이어서 하기: 같은 설정으로 이미 끝낸 폴드는 다시 학습하지 않습니다 ---
    if RESUME and drive_ckpt.is_file():
        # weights_only=True: 체크포인트를 읽을 때 임의 코드가 실행되지 않도록 막습니다.
        saved = torch.load(drive_ckpt, map_location=DEVICE, weights_only=True)
        if checkpoint_matches_config(saved):
            model = build_model()
            model.load_state_dict(saved['model'])
            model.eval()
            fold_models[fold] = model
            fold_scores[fold] = saved['val_macro_f1']
            print(f'===== fold {fold} ===== 저장된 체크포인트 재사용 '
                  f'(epoch {saved["epoch"]}, val Macro-F1 {saved["val_macro_f1"]:.5f})')
            continue
        print(f'[fold {fold}] 저장된 체크포인트가 현재 설정과 달라 새로 학습합니다.')

    print(f'\n===== fold {fold} =====  train {len(train_indices):,} / val {len(val_indices):,}')
    model, score, history, local_ckpt = fit_fold(fold, train_indices, val_indices)
    fold_models[fold] = model
    fold_scores[fold] = score
    history['fold'] = fold
    histories.append(history)

    # 폴드가 끝날 때마다 즉시 백업 — 여기서 끊겨도 앞선 폴드는 살아남습니다
    shutil.copy2(local_ckpt, drive_ckpt)
    pd.concat(histories, ignore_index=True).to_csv(
        OUTPUT_DIR / f'history_{RUN_TAG}.csv', index=False, encoding='utf-8-sig')
    print(f'  [fold {fold}] Drive 백업 완료 -> {drive_ckpt}')

history_all = pd.concat(histories, ignore_index=True) if histories else pd.DataFrame()

print(f'\n총 소요 시간: {(time.time() - overall_t0) / 60:.1f}분')
print('폴드별 best val Macro-F1:', {k: round(float(v), 5) for k, v in fold_scores.items()})
print('체크포인트 보관:', DRIVE_CKPT_DIR)

In [ ]:
# ============================================================
# 12. ablation 로그 누적
# ============================================================
# MODE='ablation' 으로 여러 설정을 돌리면 이 CSV에 계속 쌓입니다.
# 판정 규칙: fold 0 단독 차이가 0.3%p 미만이면 '판정 불가'로 보고 기본값 유지.
ablation_path = OUTPUT_DIR / 'ablation_log.csv'
row = {
    'tag': RUN_TAG, 'mode': MODE, 'backbone': BACKBONE, 'image_size': IMAGE_SIZE,
    'clahe': USE_CLAHE, 'cutmix': USE_CUTMIX, 'weight_decay': WEIGHT_DECAY,
    'label_smooth': LABEL_SMOOTH, 'epochs': EPOCHS, 'lr': LR,
    'fold0_macro_f1': fold_scores.get(0, np.nan),
    'mean_fold_macro_f1': float(np.mean(list(fold_scores.values()))),
    'recorded_at': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'),
}
log = pd.concat([pd.read_csv(ablation_path), pd.DataFrame([row])], ignore_index=True) \
      if ablation_path.is_file() else pd.DataFrame([row])
log.to_csv(ablation_path, index=False, encoding='utf-8-sig')

print('ablation 로그:', ablation_path)
display(log.sort_values('fold0_macro_f1', ascending=False).head(15))

if MODE == 'ablation':
    print('\n' + '=' * 70)
    print('ablation 모드입니다. 후보 설정을 바꿔가며 이 노트북을 반복 실행하세요.')
    print('레시피가 확정되면 MODE = "cv" 로 바꾸고 처음부터 다시 실행하면')
    print('OOF 진단과 result.csv 까지 이어집니다.')
    print('=' * 70)

## 7-1. 폴드별 오류 진단 — 어떤 클래스에서 틀리는가

5폴드가 다 끝나기를 기다리지 않고, **학습된 폴드에 대해 바로** 오류를 들여다봅니다.
(`MODE='ablation'` 이면 fold 0만, `'cv'` 면 전부)

세 각도로 봅니다.

| 표 | 답하는 질문 |
|---|---|
| 클래스별 | 어떤 표지판 종류를 못 맞히나. recall이 낮으면 놓치는 것, precision이 낮으면 남의 것을 뺏어오는 것 |
| 혼동쌍 | 무엇을 무엇으로 착각하나 |
| **트랙 단위** | **표지판 몇 개를 틀린 건가.** 30프레임이 한 표지판이므로, 39장 오답이 "표지판 1~2개를 통째로 놓친 것"일 수도, "여러 표지판에서 흐릿한 프레임만 조금씩 놓친 것"일 수도 있습니다 |

세 번째가 이 데이터 특유의 관점입니다. 처방이 갈립니다.

- **트랙 전체를 틀림** → 그 표지판 종류 자체를 못 배움. 학습 트랙이 6개뿐인 희소 클래스라면 증강 다양성 문제
- **프레임 일부만 틀림** → 표지판은 알지만 저화질·블러 프레임에서 흔들림. TTA와 열화 증강이 효과적

In [ ]:
# ============================================================
# 12-1. 폴드별 오류 진단
# ============================================================
def diagnose_fold(probs, indices, title):
    y_true = labels[indices]
    y_pred = probs.argmax(axis=1)
    track  = groups[indices]
    wrong  = y_true != y_pred

    print('=' * 74)
    print(f'{title} | {len(indices):,}장 중 {int(wrong.sum())}장 오답  '
          f'(acc {accuracy_score(y_true, y_pred):.5f}, '
          f'macro-F1 {f1_score(y_true, y_pred, average="macro"):.5f})')
    print('=' * 74)
    if not wrong.any():
        print('오답이 없습니다.')
        return

    # --- (1) 클래스별 ---------------------------------------------
    f1_all = f1_score(y_true, y_pred, average=None, labels=range(NUM_CLASSES),
                      zero_division=0)
    rows = []
    for class_id in range(NUM_CLASSES):
        is_true = y_true == class_id
        is_pred = y_pred == class_id
        if not is_true.any():
            continue
        rows.append({
            'class': class_id,
            'name': SIGN_NAMES_KO[class_id],
            'val장수': int(is_true.sum()),
            'val트랙': int(len(set(track[is_true]))),
            '틀린장수': int((y_pred[is_true] != class_id).sum()),
            'recall': float(is_pred[is_true].mean()),          # 놓친 비율의 반대
            'precision': float(is_true[is_pred].mean()) if is_pred.any() else 0.0,
            'f1': float(f1_all[class_id]),
            '학습트랙': int(tracks_per_class[class_id]),
        })
    per_class = pd.DataFrame(rows).sort_values(['f1', '틀린장수'], ascending=[True, False])
    print('\n[1] 클래스별 — F1 낮은 순 10개')
    display(per_class.head(10).round(4))

    # --- (2) 혼동쌍 -----------------------------------------------
    pairs = (pd.DataFrame({
                '정답': [f'{c:2d} {SIGN_NAMES_KO[c]}' for c in y_true[wrong]],
                '오답': [f'{c:2d} {SIGN_NAMES_KO[c]}' for c in y_pred[wrong]]})
             .value_counts().reset_index(name='건수'))
    print('\n[2] 혼동쌍 — 많이 헷갈린 순 10개')
    display(pairs.head(10))

    # --- (3) 트랙 단위 --------------------------------------------
    track_rows = []
    for track_id in sorted(set(track[wrong])):
        in_track = track == track_id
        true_class = int(y_true[in_track][0])
        bad = y_pred[in_track] != true_class
        track_rows.append({
            '트랙': track_id,
            '정답': true_class,
            '정답이름': SIGN_NAMES_KO[true_class],
            '프레임': int(in_track.sum()),
            '틀린프레임': int(bad.sum()),
            '주된오답': int(pd.Series(y_pred[in_track][bad]).mode().iloc[0]),
            '학습트랙': int(tracks_per_class[true_class]),
        })
    track_frame = pd.DataFrame(track_rows)
    track_frame['오답이름'] = [SIGN_NAMES_KO[c] for c in track_frame['주된오답']]
    track_frame = track_frame.sort_values('틀린프레임', ascending=False)

    whole = int((track_frame['틀린프레임'] == track_frame['프레임']).sum())
    partial = len(track_frame) - whole
    print(f'\n[3] 트랙 단위 — 오답이 나온 표지판 {len(track_frame)}개')
    print(f'    통째로 틀린 표지판 {whole}개 (종류 자체를 못 배움)')
    print(f'    일부 프레임만 틀린 표지판 {partial}개 (표지판은 알지만 어려운 프레임에서 흔들림)')
    display(track_frame.head(10))

for fold in sorted(fold_models):
    _, val_indices = folds[fold]
    val_loader = make_loader(
        TrafficSignDataset(paths, labels, val_indices, eval_tf), INFER_BATCH, shuffle=False)
    fold_probs, _ = predict_probs(fold_models[fold], val_loader)
    diagnose_fold(fold_probs, val_indices, f'fold {fold}')
    print()

## 8. OOF 진단 — 정직한 성적표

여기서 나오는 숫자가 **유일한 판정 기준**입니다.

- **프레임 단위 Macro-F1**: 26,010장 각각을, 그 트랙을 한 번도 못 본 모델이 예측한 결과
- **트랙 단위 Macro-F1**: 트랙 30프레임의 확률을 평균해 표지판 하나당 한 번 판정

두 값의 격차가 크면 "어려운 프레임"이 문제이고 (TTA·증강이 효과적),
작으면 "어려운 표지판 종류"가 문제입니다 (클래스별 대책이 필요).

> `MODE='ablation'` 이면 fold 0만 있어서 OOF가 불완전하므로 이후 셀은 건너뜁니다.

In [ ]:
# ============================================================
# 13. OOF 확률 구성 (TTA 포함)
# ============================================================
RUN_FULL = (MODE == 'cv')

def predict_with_tta(model, indices, targets, use_tta):
    # TTA 조합별 확률을 평균합니다. use_tta=False면 기본 변환 1회.
    specs = TTA_SPECS if use_tta else [(1.0, 0.0)]
    accumulated = None
    for scale, angle in specs:
        tf = build_eval_transform(IMAGE_SIZE, USE_CLAHE, scale, angle)
        loader = make_loader(TrafficSignDataset(paths, targets, indices, tf),
                             INFER_BATCH, shuffle=False)
        probs, _ = predict_probs(model, loader)
        accumulated = probs if accumulated is None else accumulated + probs
    return accumulated / len(specs)

if RUN_FULL:
    oof_plain = np.zeros((N, NUM_CLASSES), dtype=np.float64)
    oof_tta   = np.zeros((N, NUM_CLASSES), dtype=np.float64)

    for fold in range(N_SPLITS):
        _, val_indices = folds[fold]
        model = fold_models[fold]
        oof_plain[val_indices] = predict_with_tta(model, val_indices, labels, use_tta=False)
        oof_tta[val_indices]   = predict_with_tta(model, val_indices, labels, use_tta=True) \
                                 if USE_TTA else oof_plain[val_indices]
        print(f'  fold {fold} OOF 예측 완료')

    assert np.allclose(oof_plain.sum(axis=1), 1.0, atol=1e-4), 'OOF에 채워지지 않은 행이 있습니다'
    np.savez_compressed(OUTPUT_DIR / f'oof_{RUN_TAG}.npz',
                        oof_plain=oof_plain.astype(np.float32),
                        oof_tta=oof_tta.astype(np.float32),
                        labels=labels, groups=groups)
    print('OOF 저장:', OUTPUT_DIR / f'oof_{RUN_TAG}.npz')
else:
    print('ablation 모드 - OOF 구성을 건너뜁니다.')

In [ ]:
# ============================================================
# 14. OOF 성적표
# ============================================================
def track_level_scores(probs):
    # 같은 트랙의 30프레임 확률을 평균 → 표지판 하나당 한 번 판정
    frame = pd.DataFrame({'group': groups, 'label': labels})
    track_true, track_pred = [], []
    for group, rows in frame.groupby('group').groups.items():
        idx = np.asarray(rows)
        track_true.append(labels[idx[0]])
        track_pred.append(probs[idx].mean(axis=0).argmax())
    return np.asarray(track_true), np.asarray(track_pred)

if RUN_FULL:
    print('=' * 72)
    print(f'{"":26s}{"Accuracy":>12s}{"Macro-F1":>12s}')
    print('-' * 72)
    for name, probs in [('프레임 단위 (TTA 없음)', oof_plain), ('프레임 단위 (TTA)', oof_tta)]:
        pred = probs.argmax(axis=1)
        print(f'{name:26s}{accuracy_score(labels, pred):>12.5f}'
              f'{f1_score(labels, pred, average="macro"):>12.5f}')
    t_true, t_pred = track_level_scores(oof_tta)
    print(f'{"트랙 단위 (TTA)":26s}{accuracy_score(t_true, t_pred):>12.5f}'
          f'{f1_score(t_true, t_pred, average="macro"):>12.5f}')
    print('=' * 72)

    tta_gain = f1_score(labels, oof_tta.argmax(1), average='macro') \
             - f1_score(labels, oof_plain.argmax(1), average='macro')
    print(f'TTA 이득: {tta_gain * 100:+.3f}%p  '
          f'({"채택 권장" if tta_gain > 0 else "효과 없음 - USE_TTA=False 고려"})')
    print(f'트랙 {len(t_true)}개 중 {int((t_true != t_pred).sum())}개를 통째로 틀렸습니다.')

In [ ]:
# ============================================================
# 15. 클래스별 약점과 혼동쌍 — 다음 개선의 근거
# ============================================================
if RUN_FULL:
    oof_pred = oof_tta.argmax(axis=1)
    per_class_f1 = f1_score(labels, oof_pred, average=None, labels=range(NUM_CLASSES))

    weakest = pd.DataFrame({
        'class': range(NUM_CLASSES),
        'name': [SIGN_NAMES_KO[c] for c in range(NUM_CLASSES)],
        'tracks': [int(tracks_per_class[c]) for c in range(NUM_CLASSES)],
        'images': [int(imgs_per_class[c]) for c in range(NUM_CLASSES)],
        'f1': per_class_f1,
    }).sort_values('f1')

    print('가장 약한 클래스 12개 (Macro-F1은 이 꼬리가 결정합니다)')
    display(weakest.head(12).round({'f1': 4}))

    cm = confusion_matrix(labels, oof_pred, labels=range(NUM_CLASSES))
    np.fill_diagonal(cm, 0)
    pairs = [
        {'정답': f'{i} {SIGN_NAMES_KO[i]}', '오답': f'{j} {SIGN_NAMES_KO[j]}',
         '건수': int(cm[i, j]), '정답클래스_트랙수': int(tracks_per_class[i])}
        for i, j in zip(*np.nonzero(cm))
    ]
    top_pairs = pd.DataFrame(pairs).sort_values('건수', ascending=False).head(20)

    print('\n가장 많이 헷갈리는 쌍 상위 20')
    display(top_pairs)

    weakest.to_csv(OUTPUT_DIR / f'oof_per_class_{RUN_TAG}.csv', index=False, encoding='utf-8-sig')
    top_pairs.to_csv(OUTPUT_DIR / f'oof_confusion_pairs_{RUN_TAG}.csv', index=False, encoding='utf-8-sig')
    print('\n저장:', OUTPUT_DIR / f'oof_per_class_{RUN_TAG}.csv')

## 9. Macro-F1용 logit adjustment

클래스마다 학습 표본 수가 7배 차이 나므로 모델의 출력 확률은 다수 클래스로 치우칩니다.
확률에서 `τ × log(사전확률)` 을 빼주면 이 치우침을 되돌릴 수 있고,
Accuracy는 조금 손해 보더라도 **Macro-F1은 올라가는 경우가 많습니다.**

τ는 OOF 26,010장에서 Macro-F1이 최대가 되는 값으로 고릅니다.
다만 손실함수에 이미 클래스 가중치가 들어가 있어 이득이 없을 수도 있습니다.
**이득이 0.1%p 미만이면 τ=0(적용 안 함)으로 둡니다** — 과적합 위험을 사지 않기 위해서입니다.

In [ ]:
# ============================================================
# 16. τ 탐색
# ============================================================
SELECTED_TAU = 0.0

def apply_logit_adjustment(probs, tau, prior):
    if tau == 0.0:
        return probs
    adjusted = np.log(probs + 1e-12) - tau * np.log(prior)
    adjusted = adjusted - adjusted.max(axis=1, keepdims=True)
    exp = np.exp(adjusted)
    return exp / exp.sum(axis=1, keepdims=True)

if RUN_FULL and TUNE_TAU:
    prior = np.bincount(labels, minlength=NUM_CLASSES).astype(np.float64)
    prior = prior / prior.sum()

    rows = []
    for tau in np.round(np.arange(0.0, 1.01, 0.05), 2):
        pred = apply_logit_adjustment(oof_tta, tau, prior).argmax(axis=1)
        rows.append({'tau': tau,
                     'macro_f1': f1_score(labels, pred, average='macro'),
                     'accuracy': accuracy_score(labels, pred)})
    tau_table = pd.DataFrame(rows)

    base_f1 = tau_table.loc[tau_table['tau'] == 0.0, 'macro_f1'].iloc[0]
    best = tau_table.loc[tau_table['macro_f1'].idxmax()]
    gain = best['macro_f1'] - base_f1

    print(tau_table.to_string(index=False, float_format=lambda v: f'{v:.5f}'))
    print(f'\n최적 τ={best["tau"]} | Macro-F1 {base_f1:.5f} -> {best["macro_f1"]:.5f} ({gain * 100:+.3f}%p)')

    if gain >= 0.001:
        SELECTED_TAU = float(best['tau'])
        print(f'>> τ={SELECTED_TAU} 채택 (이득 0.1%p 이상)')
    else:
        print('>> 이득이 0.1%p 미만이라 τ=0 유지 (과적합 위험 회피)')

    tau_table.to_csv(OUTPUT_DIR / f'tau_search_{RUN_TAG}.csv', index=False, encoding='utf-8-sig')

print('최종 SELECTED_TAU =', SELECTED_TAU)

## 10. Test 추론과 제출 파일 생성

**Test 이미지는 학습에 일절 사용하지 않았습니다.** 여기서는 예측만 합니다.

### 파일 두 개가 만들어집니다

**`result.csv`** — 조교 제출용. 두 열뿐입니다.

```
00000.png,16
00001.png,1
```

**`result_confidence.csv`** — 검토용. 같은 행 순서를 유지합니다.

| 컬럼 | 뜻 |
|---|---|
| `filename` | 테스트 이미지 파일명 |
| `fold0_pred` … `fold4_pred` | **폴드별 모델 5개가 각각 내놓은 클래스** |
| `folds_agree` | 최종 예측과 같은 답을 낸 폴드 수 (1~5) |
| `final_pred` / `final_sign_name` | 5폴드 확률을 평균해 정한 **최종 분류** |
| `final_confidence` | **최종 분류의 확률값** |
| `second_pred` / `second_confidence` | 2순위 후보와 그 확률 |
| `margin` | 1순위 − 2순위. 작을수록 아슬아슬한 판단 |

### 어떻게 쓰는가

`folds_agree` 가 3 이하인 행부터 보세요.
서로 다르게 학습된 모델 5개가 갈렸다는 뜻이고, **오류는 거의 다 여기에 몰립니다.**
confidence는 모델이 자신만만하게 틀릴 수 있지만, 독립적인 5개가 갈린 것은
그 이미지가 실제로 애매하다는 훨씬 강한 신호입니다.

```python
review = pd.read_csv(confidence_path)
review[review['folds_agree'] <= 3].sort_values('margin').head(30)
```

### 제출 규칙 체크리스트
- 예측은 정수
- 제공된 템플릿의 행 순서를 그대로 유지
- header와 index 없음

In [ ]:
# ============================================================
# 17. Test 추론
# ============================================================
def read_test_filenames():
    # 템플릿이 있으면 그 순서를 그대로 따릅니다 (채점 정렬의 기준).
    if RESULT_TEMPLATE is not None:
        raw = pd.read_csv(RESULT_TEMPLATE, header=None, dtype=str)
        first = str(raw.iloc[0, 0]).strip().lower()
        names = raw.iloc[:, 0] if first.endswith(tuple(IMAGE_EXTS)) else raw.iloc[1:, 0]
        return names.astype(str).str.strip().tolist()
    return sorted(p.name for p in TEST_DIR.iterdir() if p.suffix.lower() in IMAGE_EXTS)

if RUN_FULL:
    test_names = read_test_filenames()
    test_paths = np.array([str(TEST_DIR / name) for name in test_names])
    missing = [n for n, p in zip(test_names, test_paths) if not Path(p).is_file()]
    assert not missing, f'Test 폴더에 없는 파일 {len(missing)}개, 첫 항목={missing[0]}'
    print(f'Test {len(test_names):,}장 | {N_SPLITS}폴드 x {len(TTA_SPECS) if USE_TTA else 1} TTA')

    test_indices = np.arange(len(test_names))
    test_prob = np.zeros((len(test_names), NUM_CLASSES), dtype=np.float64)
    per_fold_prob = {}

    t0 = time.time()
    for fold in range(N_SPLITS):
        specs = TTA_SPECS if USE_TTA else [(1.0, 0.0)]
        fold_prob = np.zeros_like(test_prob)
        for scale, angle in specs:
            tf = build_eval_transform(IMAGE_SIZE, USE_CLAHE, scale, angle)
            loader = make_loader(TrafficSignDataset(test_paths, None, test_indices, tf),
                                 INFER_BATCH, shuffle=False)
            probs, _ = predict_probs(fold_models[fold], loader)
            fold_prob += probs
        fold_prob /= len(specs)
        per_fold_prob[fold] = fold_prob
        test_prob += fold_prob
        print(f'  fold {fold} 추론 완료 ({time.time() - t0:.0f}s 누적)')

    test_prob /= N_SPLITS
    prior = np.bincount(labels, minlength=NUM_CLASSES).astype(np.float64)
    prior = prior / prior.sum()
    final_prob = apply_logit_adjustment(test_prob, SELECTED_TAU, prior)
    final_pred = final_prob.argmax(axis=1).astype(np.int64)

    agree = np.mean([per_fold_prob[f].argmax(1) == final_pred for f in range(N_SPLITS)])
    print(f'\n추론 완료 ({(time.time() - t0) / 60:.1f}분) | 폴드-최종 예측 일치율 {agree:.4f}')

In [ ]:
# ============================================================
# 18. result.csv 저장 + 제출 형식 검증
# ============================================================
if RUN_FULL:
    result_path     = OUTPUT_DIR / 'result.csv'
    confidence_path = OUTPUT_DIR / 'result_confidence.csv'

    # 엑셀 등이 파일을 잠갔으면 추론 결과를 잃기 전에 먼저 실패시킵니다.
    for target in (result_path, confidence_path):
        if target.exists():
            try:
                with target.open('r+', encoding='utf-8'):
                    pass
            except PermissionError as exc:
                raise PermissionError(f'다른 프로그램이 파일을 열고 있습니다: {target}') from exc

    # --- (1) result.csv : 제출용. 파일명과 정수 클래스 두 열뿐입니다 -----
    pd.DataFrame({0: test_names, 1: final_pred}).to_csv(result_path, index=False, header=False)

    # --- (2) result_confidence.csv : 검토용 -----------------------------
    # 폴드별 예측을 나란히 놓으면 5개 모델이 갈린 지점이 바로 보입니다.
    # confidence는 혼자서도 틀릴 수 있지만, 서로 다르게 학습된 5개가 갈렸다는 것은
    # 그 이미지가 진짜로 애매하다는 훨씬 강한 신호입니다.
    fold_pred = {f: per_fold_prob[f].argmax(axis=1) for f in range(N_SPLITS)}
    folds_agree = np.sum([fold_pred[f] == final_pred for f in range(N_SPLITS)], axis=0)

    # kind='stable': 확률이 정확히 같을 때 argmax와 동일하게 낮은 클래스를 고릅니다.
    # 기본 정렬은 불안정해서 동점일 때 final_pred와 어긋날 수 있습니다.
    order  = np.argsort(-final_prob, axis=1, kind='stable')[:, :2]
    values = np.take_along_axis(final_prob, order, axis=1)
    assert (order[:, 0] == final_pred).all(), '최종 예측과 1순위가 어긋납니다'

    confidence_frame = pd.DataFrame({
        'original_row': np.arange(len(test_names), dtype=np.int64),
        'filename': test_names,
    })
    for f in range(N_SPLITS):                       # fold0_pred ... fold4_pred
        confidence_frame[f'fold{f}_pred'] = fold_pred[f].astype(np.int64)
    confidence_frame['folds_agree']      = folds_agree.astype(np.int64)
    confidence_frame['final_pred']       = final_pred
    confidence_frame['final_sign_name']  = [SIGN_NAMES_KO[int(c)] for c in final_pred]
    # τ>0 이면 이 값은 logit adjustment를 적용한 뒤의 확률입니다 (실제 판단 근거).
    confidence_frame['final_confidence'] = values[:, 0]
    confidence_frame['second_pred']      = order[:, 1].astype(np.int64)
    confidence_frame['second_sign_name'] = [SIGN_NAMES_KO[int(c)] for c in order[:, 1]]
    confidence_frame['second_confidence'] = values[:, 1]
    confidence_frame['margin']           = values[:, 0] - values[:, 1]

    confidence_frame.to_csv(confidence_path, index=False,
                            float_format='%.6f', encoding='utf-8-sig')

    np.savez_compressed(OUTPUT_DIR / f'test_probs_{RUN_TAG}.npz',
                        filenames=np.asarray(test_names),
                        mean_prob=test_prob.astype(np.float32),
                        final_prob=final_prob.astype(np.float32),
                        tau=np.float32(SELECTED_TAU),
                        **{f'fold{f}': per_fold_prob[f].astype(np.float32) for f in range(N_SPLITS)})

    # --- 제출 형식 최종 검증 -----------------------------------
    check = pd.read_csv(result_path, header=None, dtype={0: str, 1: np.int64})
    assert check.shape == (len(test_names), 2), f'행/열 개수 이상: {check.shape}'
    assert check.iloc[:, 0].tolist() == test_names, '행 순서가 바뀌었습니다'
    assert check.iloc[:, 1].between(0, NUM_CLASSES - 1).all(), '0~42 범위를 벗어난 클래스'
    assert check.iloc[:, 1].dtype == np.int64, '정수가 아닙니다'
    first_line = result_path.read_text(encoding='utf-8').splitlines()[0]
    assert first_line.split(',')[0] == test_names[0], 'header 또는 index가 들어갔습니다'

    # confidence 파일도 같은 행 순서인지 확인 (두 파일을 나란히 볼 수 있어야 하므로)
    conf_check = pd.read_csv(confidence_path)
    assert len(conf_check) == len(test_names), 'confidence 파일의 행 수가 다릅니다'
    assert conf_check['filename'].tolist() == test_names, 'confidence 파일의 행 순서가 다릅니다'
    assert conf_check['final_pred'].tolist() == final_pred.tolist(), '두 파일의 예측이 다릅니다'
    assert conf_check['folds_agree'].between(1, N_SPLITS).all(), 'folds_agree 값이 이상합니다'

    print('PASS: 행 순서 유지 / 정수 / header·index 없음 / 두 파일 정합')
    print()
    print('[1] 제출 파일:', result_path)
    print('[2] 검토 파일:', confidence_path)
    print('    컬럼:', ', '.join(confidence_frame.columns))
    print()

    # 폴드 합의도 — 5개 모델이 갈린 곳이 곧 어려운 이미지입니다
    print('폴드 합의도 분포 (최종 예측과 같은 예측을 낸 폴드 수)')
    for n_agree in range(N_SPLITS, 0, -1):
        count = int((folds_agree == n_agree).sum())
        share = count / len(test_names) * 100
        mark = '   <-- 검토 대상' if n_agree <= 3 else ''
        print(f'  {n_agree}/{N_SPLITS} 일치: {count:5,}장 ({share:5.2f}%){mark}')
    review_n = int((folds_agree <= 3).sum())
    print(f'\n5개 모델이 갈린 이미지: {review_n:,}장 '
          f'({review_n / len(test_names) * 100:.2f}%) — 여기에 오류가 몰려 있습니다.')
    print(f'평균 confidence: {confidence_frame["final_confidence"].mean():.4f} | '
          f'0.9 미만: {int((confidence_frame["final_confidence"] < 0.9).sum()):,}장')
    print()
    print('예측 클래스 분포 (상위 8):')
    print(pd.Series(final_pred).map(SIGN_NAMES_KO).value_counts().head(8).to_string())

In [ ]:
# ============================================================
# 19. 최종 요약
# ============================================================
if RUN_FULL:
    summary = {
        '실행 태그': RUN_TAG,
        '백본 / 해상도': f'{BACKBONE} @ {IMAGE_SIZE}px',
        'CLAHE / CutMix': f'{USE_CLAHE} / {USE_CUTMIX}',
        '분할': f'트랙 단위 라운드로빈 {N_SPLITS}폴드 — 누수 0',
        'OOF Macro-F1 (프레임)': f'{f1_score(labels, oof_tta.argmax(1), average="macro"):.5f}',
        'OOF Accuracy (프레임)': f'{accuracy_score(labels, oof_tta.argmax(1)):.5f}',
        'OOF Macro-F1 (트랙)': f'{f1_score(*track_level_scores(oof_tta), average="macro"):.5f}',
        'TTA': f'{len(TTA_SPECS)}조합' if USE_TTA else '미사용',
        'logit adjustment τ': SELECTED_TAU,
        '제출 파일': str(result_path),
    }
    print('=' * 72)
    for key, value in summary.items():
        print(f'{key:24s}: {value}')
    print('=' * 72)
    print()
    print('다음 단계')
    print('  1. 15번 셀의 "가장 약한 클래스"와 "혼동쌍"을 확인하세요.')
    print('  2. 약한 클래스가 트랙 6개짜리에 몰려 있으면 → 증강 다양성 강화')
    print('  3. 특정 쌍(예: 제한속도 숫자)에 몰려 있으면 → 해상도 상향 또는 해당 쌍 전용 대책')
    print('  4. OOF Macro-F1이 오른 설정만 채택하세요. 제출 점수로 튜닝하면 시행횟수를 낭비합니다.')

    with open(OUTPUT_DIR / f'summary_{RUN_TAG}.json', 'w', encoding='utf-8') as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

## 11. 눈으로 확인하기 — 헷갈린 테스트 이미지 vs 클래스 대표 사진

confidence가 낮은 테스트 이미지를 왼쪽에 놓고, 오른쪽에 **모델이 고민한 두 후보 클래스의
대표 사진**을 나란히 붙여 보여줍니다. 사람 눈으로 "모델이 왜 헷갈렸는지"를 바로 볼 수 있습니다.

**대표 사진은 그 클래스에서 가장 큰(= 가장 선명한) 학습 이미지**를 씁니다.
트랙 안에서 차가 다가갈수록 표지판이 커지므로, 가장 큰 프레임이 그 클래스를 가장 잘 보여줍니다.

### 쓰는 법

```python
compare_with_class(n=12)                       # confidence 낮은 순 12장
compare_with_class(n=12, sort_by='margin')     # 1·2순위가 아슬아슬한 순
compare_with_class(n=12, only_split=True)      # 폴드가 갈린 것만
compare_with_class(n=8, start=12)              # 다음 8장 (이어서 보기)
show_all_classes()                             # 43개 클래스 대표 사진 전체 표
```

### 무엇을 볼 것인가

| 보이는 것 | 뜻 | 대응 |
|---|---|---|
| 테스트 이미지가 뭉개짐·가려짐 | 입력 자체가 어려움 | 어쩔 수 없음. 열화 증강으로 완화 |
| 두 대표 사진이 실제로 비슷 | 클래스 간 본질적 유사 | 해상도 상향, 그 쌍 전용 대책 |
| 사람 눈에는 명확한데 모델이 틀림 | 모델의 맹점 | 그 유형을 겨냥한 증강 추가 |

In [ ]:
# ============================================================
# 20. 한글 폰트 + 클래스 대표 사진 준비
# ============================================================
import matplotlib.pyplot as plt
import matplotlib.font_manager as font_manager

def setup_korean_font():
    # matplotlib 기본 폰트에는 한글이 없어 제목이 □□□로 깨집니다.
    # 이미 설치된 한글 폰트를 찾아 쓰고, 없으면 None을 돌려줍니다.
    candidates = ['NanumGothic', 'NanumBarunGothic', 'Malgun Gothic',
                  'AppleGothic', 'Noto Sans CJK KR', 'Noto Sans KR']
    available = {f.name for f in font_manager.fontManager.ttflist}
    for name in candidates:
        if name in available:
            plt.rc('font', family=name)
            plt.rc('axes', unicode_minus=False)
            return name
    return None

KOREAN_FONT = setup_korean_font()
if KOREAN_FONT:
    print('한글 폰트:', KOREAN_FONT)
else:
    print('한글 폰트가 없어 그림 제목은 영문/번호로 표시합니다.')
    print('한글로 보시려면 새 셀에서 아래를 실행하고 런타임을 재시작하세요:')
    print('  !apt-get -qq install -y fonts-nanum && fc-cache -fv')

def T(korean, english):
    # 한글 폰트가 없으면 영문으로 대체 (□□□ 방지)
    return korean if KOREAN_FONT else english

def class_label(class_id):
    return f'{class_id} {SIGN_NAMES_KO[class_id]}' if KOREAN_FONT else f'class {class_id}'

# --- 클래스별 대표 사진 고르기 -----------------------------------
def build_class_representatives():
    # 가장 큰 이미지 = 차가 가장 가까이서 찍은 프레임 = 가장 선명한 관측
    # Image.open은 헤더만 읽으므로 .size 조회는 빠릅니다.
    representatives = {}
    for class_id in range(NUM_CLASSES):
        best_path, best_area = None, -1
        for i in np.where(labels == class_id)[0]:
            with Image.open(paths[i]) as image:
                area = image.size[0] * image.size[1]
            if area > best_area:
                best_area, best_path = area, paths[i]
        representatives[class_id] = best_path
    return representatives

_t0 = time.time()
CLASS_REP = build_class_representatives()
assert len(CLASS_REP) == NUM_CLASSES and all(CLASS_REP.values())
print(f'클래스 대표 사진 {len(CLASS_REP)}개 선정 완료 ({time.time() - _t0:.0f}초)')

In [ ]:
# ============================================================
# 21. 비교 뷰어
# ============================================================
def _draw(ax, image_path, title, highlight=None):
    with Image.open(image_path) as image:
        ax.imshow(image.convert('RGB'))
    ax.set_title(title, fontsize=9, loc='left')
    ax.set_xticks([]); ax.set_yticks([])
    for side in ax.spines.values():
        side.set_visible(highlight is not None)
        if highlight is not None:
            side.set_color(highlight); side.set_linewidth(2.5)

def show_all_classes(columns=9):
    # 43개 클래스 대표 사진 한눈에 보기 (혼동쌍을 눈으로 확인할 때 참고표)
    rows = math.ceil(NUM_CLASSES / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(columns * 1.7, rows * 2.0))
    for ax in axes.ravel():
        ax.axis('off')
    for class_id in range(NUM_CLASSES):
        ax = axes.ravel()[class_id]
        ax.axis('on')
        _draw(ax, CLASS_REP[class_id], class_label(class_id))
    fig.suptitle(T('클래스 대표 사진 43개', 'Class reference (43 classes)'), fontsize=13)
    plt.tight_layout()
    plt.show()

def compare_with_class(n=12, sort_by='confidence', only_split=False, start=0,
                       filenames=None):
    """
    sort_by  : 'confidence' (최종 확률 낮은 순) | 'margin' (1·2순위 차이 작은 순)
    only_split: True면 폴드가 갈린 이미지만
    filenames : 특정 파일명만 보고 싶을 때 리스트로 지정
    """
    frame = pd.read_csv(confidence_path)
    if filenames is not None:
        frame = frame[frame['filename'].isin(
            [filenames] if isinstance(filenames, str) else filenames)]
    if only_split:
        frame = frame[frame['folds_agree'] <= N_SPLITS - 2]
    frame = frame.sort_values(
        'final_confidence' if sort_by == 'confidence' else 'margin'
    ).iloc[start:start + n]

    if frame.empty:
        print('조건에 해당하는 이미지가 없습니다.')
        return frame

    fig, axes = plt.subplots(len(frame), 3, figsize=(10.5, 3.5 * len(frame)))
    axes = np.atleast_2d(axes)

    for r, (_, row) in enumerate(frame.iterrows()):
        first, second = int(row['final_pred']), int(row['second_pred'])
        votes = ' '.join(str(int(row[f'fold{f}_pred'])) for f in range(N_SPLITS))
        _draw(axes[r, 0], TEST_DIR / row['filename'],
              f"TEST  {row['filename']}\n"
              f"{T('폴드예측', 'folds')}  {votes}\n"
              f"{T('합의', 'agree')} {int(row['folds_agree'])}/{N_SPLITS}",
              highlight='red')
        _draw(axes[r, 1], CLASS_REP[first],
              f"{T('1순위', 'top-1')}  {class_label(first)}\n"
              f"{T('확률', 'prob')} {row['final_confidence']:.3f}",
              highlight='tab:green')
        _draw(axes[r, 2], CLASS_REP[second],
              f"{T('2순위', 'top-2')}  {class_label(second)}\n"
              f"{T('확률', 'prob')} {row['second_confidence']:.3f}"
              f"   (margin {row['margin']:.3f})",
              highlight='tab:orange')

    plt.tight_layout()
    plt.show()

    columns = ['filename'] + [f'fold{f}_pred' for f in range(N_SPLITS)] + \
              ['folds_agree', 'final_pred', 'final_sign_name', 'final_confidence',
               'second_pred', 'second_sign_name', 'second_confidence', 'margin']
    display(frame[columns].reset_index(drop=True).round(4))
    return frame

if RUN_FULL:
    print(T('43개 클래스 대표 사진', 'Class reference'))
    show_all_classes()
    print()
    print(T('confidence 가장 낮은 12장 — 왼쪽이 테스트 이미지, 오른쪽 둘이 후보 클래스',
            'Lowest-confidence 12 test images vs candidate class references'))
    _low = compare_with_class(n=12, sort_by='confidence')
else:
    print('MODE="cv" 로 추론까지 끝낸 뒤 사용할 수 있습니다.')